In [1]:
from langchain_core.messages import BaseMessage,HumanMessage,AIMessage
from langgraph.graph import StateGraph,START,END
from typing import List, TypedDict, Union, Any
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()
llm=ChatGroq(
    model="groq/compound",
    api_key=os.getenv("GROQ_API_KEY")
)

In [3]:
class AgentState(TypedDict):
    message: List[Union[HumanMessage,AIMessage]]

In [7]:
def process(state: AgentState) -> AgentState:
    response = llm.invoke(state["message"])
    state["message"].append(AIMessage(content=response.content))
    print(f"AI: {response.content}")
    print(f"current state: {state['message']}")
    return state

In [8]:
graph=StateGraph(AgentState)
graph.add_node("process", process)
graph.add_edge(START, "process")
graph.add_edge("process", END)

agent=graph.compile()

In [18]:
conversation_history = []
while True:
    user_input = input("User: ")
    print(f"User: {user_input}")
    if user_input.lower() == "exit":
        break
    else:
        conversation_history.append(HumanMessage(content=user_input))
        output = agent.invoke({"message": conversation_history})
        conversation_history = output["message"]
        print(f"Assistant: {output['message'][-1].content}")
        #print(f"current state conv: {conversation_history}")

User: hii
AI: Hi there! How can I help you today?
current state: [HumanMessage(content='hii', additional_kwargs={}, response_metadata={}), AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
Assistant: Hi there! How can I help you today?
User: what??
AI: I’m sorry—could you let me know what you’d like help with?
current state: [HumanMessage(content='hii', additional_kwargs={}, response_metadata={}), AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='what??', additional_kwargs={}, response_metadata={}), AIMessage(content='I’m sorry—could you let me know what you’d like help with?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
Assistant: I’m sorry—could you let me know what you’d like help with?
User: exit


In [19]:
with open("Results/conversation_history.txt", "w") as f:
    f.write("Conversation History:\n")
    for message in conversation_history:
        if isinstance(message, HumanMessage):
            f.write(f"User: {message.content}\n")
        elif isinstance(message, AIMessage):
            f.write(f"Assistant: {message.content}\n")
    f.write("End of Conversation\n")
            
print("Conversation history saved to conversation_history.txt") 

Conversation history saved to conversation_history.txt
